In [1]:
import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '5'
import argparse
import numpy as np
import plyfile,copy
import pandas as pd
import tifffile as tiff
import torch

from scipy import io
import os,shutil
join = os.path.join
from PIL import Image
import matplotlib.pyplot as plt
import shutil
from scipy.signal import find_peaks
import open3d as o3d
from matplotlib.colors import Normalize
from scipy.interpolate import CubicSpline, Rbf
import SimpleITK as sitk

from geomdl import NURBS, utilities
from scipy.ndimage import convolve, gaussian_filter
import tigre
import tigre.algorithms as algs

import open3d as o3d

from scipy.interpolate import CubicSpline
from mpl_toolkits.mplot3d import Axes3D

import nibabel as nib

import random
import tqdm

import SimpleITK as sitk # 推荐用于医学图像处理
from skimage.morphology import skeletonize_3d
from skimage.measure import regionprops,label

import json

import nibabel as nib
import numpy as np
from scipy.ndimage import zoom

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## 其他

### 存储查找表

In [3]:
import json
BRAIN_VESSEL_LABELS_MR = {
    # 颅内动脉及静脉窦 (IDX 0-34, 43个)
    0:  {"LABEL": "Clear Label", "FULL_NAME_EN": "N/A", "FULL_NAME_ZH": "清除标签 / 背景", "TYPE": "Background"},
    1:  {"LABEL": "BA", "FULL_NAME_EN": "Basilar Artery", "FULL_NAME_ZH": "基底动脉", "TYPE": "Artery (Posterior)"},
    2:  {"LABEL": "R-P1P2", "FULL_NAME_EN": "Right Posterior Cerebral Artery, segments P1 & P2", "FULL_NAME_ZH": "右侧大脑后动脉 P1 和 P2 段", "TYPE": "Artery (Posterior)"},
    3:  {"LABEL": "L-P1P2", "FULL_NAME_EN": "Left Posterior Cerebral Artery, segments P1 & P2", "FULL_NAME_ZH": "左侧大脑后动脉 P1 和 P2 段", "TYPE": "Artery (Posterior)"},
    4:  {"LABEL": "R-ICA", "FULL_NAME_EN": "Right Internal Carotid Artery", "FULL_NAME_ZH": "右侧颈内动脉", "TYPE": "Artery (Anterior)"},
    5:  {"LABEL": "R-M1", "FULL_NAME_EN": "Right Middle Cerebral Artery, segment M1", "FULL_NAME_ZH": "右侧大脑中动脉 M1 段", "TYPE": "Artery (Anterior)"},
    6:  {"LABEL": "L-ICA", "FULL_NAME_EN": "Left Internal Carotid Artery", "FULL_NAME_ZH": "左侧颈内动脉", "TYPE": "Artery (Anterior)"},
    7:  {"LABEL": "L-M1", "FULL_NAME_EN": "Left Middle Cerebral Artery, segment M1", "FULL_NAME_ZH": "左侧大脑中动脉 M1 段", "TYPE": "Artery (Anterior)"},
    8:  {"LABEL": "R-Pcom", "FULL_NAME_EN": "Right Posterior Communicating Artery", "FULL_NAME_ZH": "右侧后交通动脉", "TYPE": "Artery (Communicating)"},
    9:  {"LABEL": "L-Pcom", "FULL_NAME_EN": "Left Posterior Communicating Artery", "FULL_NAME_ZH": "左侧后交通动脉", "TYPE": "Artery (Communicating)"},
    10: {"LABEL": "Acom", "FULL_NAME_EN": "Anterior Communicating Artery", "FULL_NAME_ZH": "前交通动脉", "TYPE": "Artery (Communicating)"},
    11: {"LABEL": "R-A1A2", "FULL_NAME_EN": "Right Anterior Cerebral Artery, segments A1 & A2", "FULL_NAME_ZH": "右侧大脑前动脉 A1 和 A2 段", "TYPE": "Artery (Anterior)"},
    12: {"LABEL": "L-A1A2", "FULL_NAME_EN": "Left Anterior Cerebral Artery, segments A1 & A2", "FULL_NAME_ZH": "左侧大脑前动脉 A1 和 A2 段", "TYPE": "Artery (Anterior)"},
    13: {"LABEL": "R-A3", "FULL_NAME_EN": "Right Anterior Cerebral Artery, segment A3", "FULL_NAME_ZH": "右侧大脑前动脉 A3 段", "TYPE": "Artery (Anterior)"},
    14: {"LABEL": "L-A3", "FULL_NAME_EN": "Left Anterior Cerebral Artery, segment A3", "FULL_NAME_ZH": "左侧大脑前动脉 A3 段", "TYPE": "Artery (Anterior)"},
    15: {"LABEL": "3rd-A2", "FULL_NAME_EN": "3rd Anterior Cerebral Artery, segment A2 (Variant)", "FULL_NAME_ZH": "第三大脑前动脉 A2 段 (变异)", "TYPE": "Artery (Variant)"},
    16: {"LABEL": "3rd-A3", "FULL_NAME_EN": "3rd Anterior Cerebral Artery, segment A3 (Variant)", "FULL_NAME_ZH": "第三大脑前动脉 A3 段 (变异)", "TYPE": "Artery (Variant)"},
    17: {"LABEL": "R-M2", "FULL_NAME_EN": "Right Middle Cerebral Artery, segment M2", "FULL_NAME_ZH": "右侧大脑中动脉 M2 段", "TYPE": "Artery (Anterior)"},
    18: {"LABEL": "R-M3", "FULL_NAME_EN": "Right Middle Cerebral Artery, segment M3", "FULL_NAME_ZH": "右侧大脑中动脉 M3 段", "TYPE": "Artery (Anterior)"},
    19: {"LABEL": "L-M2", "FULL_NAME_EN": "Left Middle Cerebral Artery, segment M2", "FULL_NAME_ZH": "左侧大脑中动脉 M2 段", "TYPE": "Artery (Anterior)"},
    20: {"LABEL": "L-M3", "FULL_NAME_EN": "Left Middle Cerebral Artery, segment M3", "FULL_NAME_ZH": "左侧大脑中动脉 M3 段", "TYPE": "Artery (Anterior)"},
    21: {"LABEL": "R-P3P4", "FULL_NAME_EN": "Right Posterior Cerebral Artery, segments P3 & P4", "FULL_NAME_ZH": "右侧大脑后动脉 P3 和 P4 段", "TYPE": "Artery (Posterior)"},
    22: {"LABEL": "L-P3P4", "FULL_NAME_EN": "Left Posterior Cerebral Artery, segments P3 & P4", "FULL_NAME_ZH": "左侧大脑后动脉 P3 和 P4 段", "TYPE": "Artery (Posterior)"},
    23: {"LABEL": "R-VA", "FULL_NAME_EN": "Right Vertebral Artery", "FULL_NAME_ZH": "右侧椎动脉", "TYPE": "Artery (Posterior)"},
    24: {"LABEL": "L-VA", "FULL_NAME_EN": "Left Vertebral Artery", "FULL_NAME_ZH": "左侧椎动脉", "TYPE": "Artery (Posterior)"},
    25: {"LABEL": "R-SCA", "FULL_NAME_EN": "Right Superior Cerebellar Artery", "FULL_NAME_ZH": "右侧小脑上动脉", "TYPE": "Artery (Cerebellar)"},
    26: {"LABEL": "L-SCA", "FULL_NAME_EN": "Left Superior Cerebellar Artery", "FULL_NAME_ZH": "左侧小脑上动脉", "TYPE": "Artery (Cerebellar)"},
    27: {"LABEL": "R-AICA", "FULL_NAME_EN": "Right Anterior Inferior Cerebellar Artery", "FULL_NAME_ZH": "右侧小脑前下动脉", "TYPE": "Artery (Cerebellar)"},
    28: {"LABEL": "L-AICA", "FULL_NAME_EN": "Left Anterior Inferior Cerebellar Artery", "FULL_NAME_ZH": "左侧小脑前下动脉", "TYPE": "Artery (Cerebellar)"},
    29: {"LABEL": "R-PICA", "FULL_NAME_EN": "Right Posterior Inferior Cerebellar Artery", "FULL_NAME_ZH": "右侧小脑后下动脉", "TYPE": "Artery (Cerebellar)"},
    30: {"LABEL": "L-PICA", "FULL_NAME_EN": "Left Posterior Inferior Cerebellar Artery", "FULL_NAME_ZH": "左侧小脑后下动脉", "TYPE": "Artery (Cerebellar)"},
    31: {"LABEL": "R-AChA", "FULL_NAME_EN": "Right Anterior Choroidal Artery", "FULL_NAME_ZH": "右侧脉络膜前动脉", "TYPE": "Artery (Deep)"},
    32: {"LABEL": "L-AChA", "FULL_NAME_EN": "Left Anterior Choroidal Artery", "FULL_NAME_ZH": "左侧脉络膜前动脉", "TYPE": "Artery (Deep)"},
    33: {"LABEL": "R-OA", "FULL_NAME_EN": "Right Ophthalmic Artery", "FULL_NAME_ZH": "右侧眼动脉", "TYPE": "Artery (Intracranial)"},
    34: {"LABEL": "L-OA", "FULL_NAME_EN": "Left Ophthalmic Artery", "FULL_NAME_ZH": "左侧眼动脉", "TYPE": "Artery (Intracranial)"},
    
    # 颅外血管及深静脉 [MRI特有的标签]
    35: {"LABEL": "R-ECA", "FULL_NAME_EN": "Right External Carotid Artery", "FULL_NAME_ZH": "右侧颈外动脉", "TYPE": "Artery (Extracranial)"},
    36: {"LABEL": "L-ECA", "FULL_NAME_EN": "Left External Carotid Artery", "FULL_NAME_ZH": "左侧颈外动脉", "TYPE": "Artery (Extracranial)"},
    37: {"LABEL": "R-STA", "FULL_NAME_EN": "Right Superficial Temporal Artery", "FULL_NAME_ZH": "右侧颞浅动脉", "TYPE": "Artery (Extracranial)"},
    38: {"LABEL": "L-STA", "FULL_NAME_EN": "Left Superficial Temporal Artery", "FULL_NAME_ZH": "左侧颞浅动脉", "TYPE": "Artery (Extracranial)"},
    39: {"LABEL": "R-MaxA", "FULL_NAME_EN": "Right Maxillary Artery", "FULL_NAME_ZH": "右侧上颌动脉", "TYPE": "Artery (Extracranial)"},
    40: {"LABEL": "L-MaxA", "FULL_NAME_EN": "Left Maxillary Artery", "FULL_NAME_ZH": "左侧上颌动脉", "TYPE": "Artery (Extracranial)"},
    41: {"LABEL": "R-MMA", "FULL_NAME_EN": "Right Middle Meningeal Artery", "FULL_NAME_ZH": "右侧脑膜中动脉", "TYPE": "Artery (Extracranial)"},
    42: {"LABEL": "L-MMA", "FULL_NAME_EN": "Left Middle Meningeal Artery", "FULL_NAME_ZH": "左侧脑膜中动脉", "TYPE": "Artery (Extracranial)"},
    # 注意：如果您的原始数据中包含 VoG (35), StS (36), SSS (40) 等静脉标签，
    # 但新的列表用动脉覆盖了这些 IDX，那么请以新的列表为准。
    # 这里我们假设您已经更新了标签文件，用新的动脉标签替换了旧的静脉标签。
}
with open('/data1/liuyang/stent/interior/data/gen_data_utils/BRAIN_VESSEL_LABELS_MR.json', 'w') as f:
    json.dump(BRAIN_VESSEL_LABELS_MR, f, indent=4)

### costa数据集重命名

In [ ]:
## 训练数据-图像
# src_path = '/data1/liuyang/stent/interior/data/costa/111/COSTA/COSTA-Dataset-v1/imagesTr'
# save_dir = '/data1/liuyang/stent/interior/data/costa/111/COSTA/rename_costa/imagesTr'

## 训练数据-标签
# src_path = '/data1/liuyang/stent/interior/data/costa/111/COSTA/COSTA-Dataset-v1/labelsTr'
# save_dir = '/data1/liuyang/stent/interior/data/costa/111/COSTA/rename_costa/labelsTr'

## 测试数据-图像
# src_path = '/data1/liuyang/stent/interior/data/costa/111/COSTA/COSTA-Dataset-v1/labelsTs'
# save_dir = '/data1/liuyang/stent/interior/data/costa/111/COSTA/rename_costa/labelsTs'

## 测试数据-标签
src_path = '/data1/liuyang/stent/interior/data/costa/111/COSTA/COSTA-Dataset-v1/imagesTs'
save_dir = '/data1/liuyang/stent/interior/data/costa/111/COSTA/rename_costa/imagesTs'

adam_save_dir = join(save_dir, i)
os.makedirs(save_dir, exist_ok=True)

for root,dirs, files in os.walk(src_path):
    for i in files:
        if(i.rfind('ICBM')!=-1):
            base_name = 'ICBM'
            number = str(int(i[2:6])).zfill(3)
            save_name = base_name + '_' + number + '.nii.gz'
            shutil.copy(join(root, i),join(save_dir, save_name))
        elif(i.rfind('HH')!=-1):
            base_name = 'IXI_HH'
            number = str(int(i[3:6])).zfill(3)
            save_name = base_name + '_' + number + '.nii.gz'
            shutil.copy(join(root, i),join(save_dir, save_name))
        elif(i.rfind('Guys')!=-1):
            base_name = 'IXI_Guys'
            number = str(int(i[3:6])).zfill(3)
            save_name = base_name + '_' + number + '.nii.gz'
            shutil.copy(join(root, i),join(save_dir, save_name))
        elif(i.rfind('IOP')!=-1):
            base_name = 'IXI_IOP'
            number = str(int(i[3:6])).zfill(3)
            save_name = base_name + '_' + number + '.nii.gz'
            shutil.copy(join(root, i),join(save_dir, save_name))
        elif(i.rfind('LocH')!=-1):
            base_name = 'LOCH'
            number = str(int(i[6:9])).zfill(3)
            save_name = base_name + '_' + number + '.nii.gz'
            shutil.copy(join(root, i),join(save_dir, save_name))
        elif(i.rfind('ADAM')!=-1):
          base_name = 'ADAM'
          index = i.split('-')[0]
        #   index = index.replace('B', '0')
        #   index = index.replace('F', '1')
          number = str(index[3:]).zfill(3)
          save_name = base_name + '_' + number + '.nii.gz'
          shutil.copy(join(root, i),join(save_dir, save_name))

In [ ]:
# ## 把训练集的文件名修正一下
# src_path = '/data1/liuyang/stent/interior/data/costa/labelTr'
# save_dir = '/data1/liuyang/stent/interior/data/costa/root_data/train_data'
# os.makedirs(save_dir, exist_ok=True)

# for root,dirs, files in os.walk(src_path):
#     for i in files:
#         if(i.rfind('ICBM')!=-1):
#             base_name = 'ICBM'
#             number = str(int(i[2:6])).zfill(3)
#             save_name = base_name + '_' + number + '.nii.gz'
#             shutil.copy(join(root, i),join(save_dir, save_name))
#         elif(i.rfind('HH')!=-1):
#             base_name = 'IXI_HH'
#             number = str(int(i[3:6])).zfill(3)
#             save_name = base_name + '_' + number + '.nii.gz'
#             shutil.copy(join(root, i),join(save_dir, save_name))
#         elif(i.rfind('Guys')!=-1):
#             base_name = 'IXI_Guys'
#             number = str(int(i[3:6])).zfill(3)
#             save_name = base_name + '_' + number + '.nii.gz'
#             shutil.copy(join(root, i),join(save_dir, save_name))
#         elif(i.rfind('IOP')!=-1):
#             base_name = 'IXI_IOP'
#             number = str(int(i[3:6])).zfill(3)
#             save_name = base_name + '_' + number + '.nii.gz'
#             shutil.copy(join(root, i),join(save_dir, save_name))
#         elif(i.rfind('LocH')!=-1):
#             base_name = 'LOCH'
#             number = str(int(i[6:9])).zfill(3)
#             save_name = base_name + '_' + number + '.nii.gz'
#             shutil.copy(join(root, i),join(save_dir, save_name))

# ## 把测试集的文件名修正一下
# src_path = '/data1/liuyang/stent/interior/data/costa/labelTs'
# save_dir = '/data1/liuyang/stent/interior/data/costa/root_data/test_data'
# for i in os.listdir(src_path):
#     if(i.rfind('ICBM')!=-1):
#         base_name = 'ICBM'
#         number = str(int(i[2:6])).zfill(3)
#         save_name = base_name + '_' + number + '.nii.gz'
#         shutil.copy(join(src_path, i),join(save_dir, save_name))
#     elif(i.rfind('HH')!=-1):
#         base_name = 'IXI_HH'
#         number = str(int(i[3:6])).zfill(3)
#         save_name = base_name + '_' + number + '.nii.gz'
#         shutil.copy(join(src_path, i),join(save_dir, save_name))
#     elif(i.rfind('Guys')!=-1):
#         base_name = 'IXI_Guys'
#         number = str(int(i[3:6])).zfill(3)
#         save_name = base_name + '_' + number + '.nii.gz'
#         shutil.copy(join(src_path, i),join(save_dir, save_name))
#     elif(i.rfind('IOP')!=-1):
#         base_name = 'IXI_IOP'
#         number = str(int(i[3:6])).zfill(3)
#         save_name = base_name + '_' + number + '.nii.gz'
#         shutil.copy(join(src_path, i),join(save_dir, save_name))
#     elif(i.rfind('LocH')!=-1):
#         base_name = 'LOCH'
#         number = str(int(i[6:9])).zfill(3)
#         save_name = base_name + '_' + number + '.nii.gz'
#         shutil.copy(join(src_path, i),join(save_dir, save_name))

### topbrain数据集重命名

In [ ]:
image_dir = '/data1/liuyang/stent/DTE/data/topbrain/imagesTr'
label_dir = '/data1/liuyang/stent/DTE/data/topbrain/labelsTr'

for i in os.listdir(image_dir):
  os.rename(join(image_dir, i), join(image_dir, i.split('topcow_')[-1].split('_0000')[0]+'.nii.gz'))

for i in os.listdir(label_dir):
  os.rename(join(label_dir, i), join(label_dir, i.split('topcow_')[-1]))


In [ ]:
# image_dir = '/data1/liuyang/stent/DTE/data/topbrain/train_data/image'
# for i in os.listdir(image_dir):
#   os.rename(join(image_dir, i), join(image_dir, i.split('.nii.gz.nii.gz.nii.gz')[0]+'.nii.gz'))

### 其它

In [29]:
for i in os.listdir('/data1/liuyang/stent/interior/data/costa/src_data/radius-64/train_data/vol_gt'):
    if(not i.endswith('.nii.gz')):
        print(i)

In [ ]:
file_path = '/data1/liuyang/stent/interior/data/costa/src_data/radius-64/train_data/proj_gt/IXI_Guys_074_sample_00.npz'
np.load(file_path)

In [37]:
len(os.listdir('/data1/liuyang/stent/interior/data/costa/trun_data/radius-50/train_data/25_4-fdkNoFilter/vol_train'))

11747

In [3]:
x_shape, y_shape, z_shape = 0, 0, 0
resample_dir = '/data1/liuyang/stent/interior/data/costa/helper/train_data/voxel_0.2/resample'
for i in os.listdir(resample_dir):
    nib_shape = nib.load(join(resample_dir, i)).shape
    x_shape += nib_shape[0]
    y_shape += nib_shape[1]
    z_shape += nib_shape[2]
print(x_shape/len(os.listdir(resample_dir)),y_shape/len(os.listdir(resample_dir)), z_shape/len(os.listdir(resample_dir)))

677.2063492063492 860.5555555555555 445.4550264550264


In [4]:
x_shape, y_shape, z_shape = 0, 0, 0
resample_dir = '/data1/liuyang/stent/interior/data/costa/helper/train_data/voxel_0.15/resample'
for i in os.listdir(resample_dir):
    nib_shape = nib.load(join(resample_dir, i)).shape
    x_shape += nib_shape[0]
    y_shape += nib_shape[1]
    z_shape += nib_shape[2]
print(x_shape/len(os.listdir(resample_dir)),y_shape/len(os.listdir(resample_dir)), z_shape/len(os.listdir(resample_dir)))

902.9153439153439 1147.4021164021165 593.8253968253969


In [5]:
x_shape, y_shape, z_shape = 0, 0, 0
resample_dir = '/data1/liuyang/stent/interior/data/costa/helper/train_data/voxel_0.15/resample'
for i in os.listdir(resample_dir):
    nib_shape = nib.load(join(resample_dir, i)).shape
    if(nib_shape[0] > x_shape):
        x_shape = nib_shape[0]
    if(nib_shape[1] > y_shape):
        y_shape = nib_shape[1]
    if(nib_shape[2] > z_shape):
        z_shape = nib_shape[2]
print(x_shape, y_shape, z_shape)

1011 1300 1002


In [6]:
x_shape, y_shape, z_shape = 0, 0, 0
resample_dir = '/data1/liuyang/stent/interior/data/costa/helper/train_data/voxel_0.2/resample'
for i in os.listdir(resample_dir):
    nib_shape = nib.load(join(resample_dir, i)).shape
    if(nib_shape[0] > x_shape):
        x_shape = nib_shape[0]
    if(nib_shape[1] > y_shape):
        y_shape = nib_shape[1]
    if(nib_shape[2] > z_shape):
        z_shape = nib_shape[2]
print(x_shape, y_shape, z_shape)

758 975 751


In [ ]:
def check_trun_data_complexity(trun_data_save_dir):
    all_dict = []
    not_finish = []
    for i in os.listdir(trun_data_save_dir):
        # single_dir = join(trun_data_save_dir, i)
        # print(len(os.listdir(single_dir)))

        # prev_size = -1
        # for j in os.listdir(single_dir):
        #     file_size = os.path.getsize(join(single_dir, j))
        #     if(prev_size==-1):
        #         prev_size = file_size
        #     else:
        #         assert np.abs(file_size - prev_size * 3/4) > 0

        single_dir = join(trun_data_save_dir, i)
        all_num = dict()
        for j in os.listdir(single_dir):
            idx = j.split("_sample_")[0] + "_sample_" + j.split("_sample_")[1].split("_")[0]
            if(all_num.get(idx) is None):
                all_num[idx] = {'num':1, 'name': join(i,j)}
            else:
                all_num[idx]['num'] += 1
        all_dict.append(all_num)
    
    for i in all_dict:
        for j in i:
            if(i[j]['num']!=3):
                # print(i[j]['name'])
                # print(j)
                not_finish.append(j)
    return set(all_dict[0].keys()) - set(not_finish)



In [35]:
all_dict.keys()

NameError: name 'all_dict' is not defined

In [37]:
trun_data_save_dir = '/data1/liuyang/stent/interior/data/corr_data/data/voxel_0.2/trun_data/radius-64/corr_stent/train_data/60_4-fdkNoFilter'
allo_dict = check_trun_data_complexity(trun_data_save_dir)

In [20]:
len(os.listdir(trun_data_save_dir+'/vol_gt'))

3798

In [42]:
list(allo_dict[0].keys())

['ICBM_029_sample_02',
 'IXI_Guys_179_sample_00',
 'IXI_HH_248_sample_02',
 'IXI_HH_280_sample_06',
 'IXI_Guys_207_sample_06',
 'IXI_Guys_197_sample_04',
 'IXI_HH_104_sample_06',
 'IXI_Guys_135_sample_01',
 'IXI_HH_274_sample_03',
 'ICBM_019_sample_00',
 'IXI_HH_278_sample_02',
 'ICBM_017_sample_02',
 'IXI_HH_137_sample_04',
 'IXI_HH_160_sample_01',
 'IXI_HH_277_sample_05',
 'IXI_HH_173_sample_06',
 'IXI_Guys_140_sample_09',
 'IXI_HH_168_sample_07',
 'IXI_Guys_106_sample_00',
 'IXI_HH_242_sample_06',
 'IXI_Guys_087_sample_02',
 'IXI_Guys_091_sample_09',
 'IXI_Guys_189_sample_00',
 'IXI_Guys_107_sample_09',
 'IXI_HH_163_sample_09',
 'IXI_HH_136_sample_00',
 'IXI_HH_130_sample_08',
 'IXI_HH_128_sample_08',
 'IXI_HH_202_sample_05',
 'IXI_Guys_111_sample_00',
 'IXI_HH_214_sample_05',
 'IXI_Guys_070_sample_02',
 'IXI_HH_160_sample_04',
 'IXI_Guys_166_sample_03',
 'ICBM_020_sample_00',
 'IXI_Guys_076_sample_07',
 'IXI_Guys_087_sample_01',
 'IXI_Guys_170_sample_00',
 'ICBM_037_sample_01',
 'I

In [6]:
import pickle
with open('/data1/liuyang/stent/interior/data/corr_data/data/voxel_0.2/src_data/radius-64/corr_stent/train_data/config/ICBM_011.pickle', 'rb') as f:
  config_data = pickle.load(f)
config_data

{0: {'center': [255.0, 549.0, 141.0],
  'border': [204.12085505125438,
   306.258292894859,
   533.2078195815612,
   565.701047943406,
   128.34230089326164,
   153.83531529493374],
  'stent_name': 'wLikeStent'},
 1: {'center': [488.5, 534.0, 150.0],
  'border': [435.2940767739985,
   542.3996023989498,
   521.9682016588995,
   546.3149615102461,
   138.24539100215569,
   162.52813773223852],
  'stent_name': 'stent4'},
 2: {'center': [464.5, 534.5, 144.0],
  'border': [414.06231418011276,
   515.934547344287,
   523.3300404440656,
   546.6119431791234,
   126.12861615063697,
   162.1892284704206],
  'stent_name': 'sinStent'},
 3: {'center': [388.0, 574.0, 161.5],
  'border': [356.7861864217004,
   419.31810505596286,
   535.8654273938042,
   612.2536794591638,
   144.5946487996602,
   178.74402726699378],
  'stent_name': 'ringStent'},
 4: {'center': [445.5, 528.0, 133.0],
  'border': [403.2265878051364,
   488.73529761898357,
   513.5819712207286,
   542.454439817272,
   103.9496775287

{0: {'center': [236.0, 588.5, 196.5],
  'border': [199.91273823014302,
   272.87207794644456,
   556.443883091446,
   620.6626096533115,
   171.12694737865985,
   222.00111250003917],
  'stent_name': 'ringStent'},
 1: {'center': [467.5, 591.5, 187.5],
  'border': [428.3583393743835,
   506.9681794646142,
   560.2700069335414,
   623.4119399955314,
   173.1359484009845,
   202.35093155712534],
  'stent_name': 'stent4'},
 2: {'center': [479.0, 471.0, 57.0],
  'border': [464.30398450707276,
   494.2322094163401,
   442.8809163037389,
   499.46301463130595,
   12.591039531376284,
   102.28666028875355],
  'stent_name': 'stent4'},
 3: {'center': [335.0, 470.5, 75.5],
  'border': [301.86523541258947,
   368.93333659381216,
   448.7483120531891,
   493.07661837812145,
   55.82883964174009,
   95.69552218822197],
  'stent_name': 'stent2'},
 4: {'center': [264.0, 571.0, 193.5],
  'border': [224.35645757233027,
   303.7016005995788,
   546.1141187648752,
   596.4120210298765,
   165.274243042427

In [ ]:
len(os.listdir('/data1/liuyang/stent/interior/data/corr_data/data/costa/voxel_0.2/src_data/radius-64/corr_stent/test_data/vol_gt'))

460

: 

In [ ]:
# root_path = '/data1/liuyang/stent/DTR/data_3'
# save_dir = '/data1/liuyang/stent/DTR/to_transfer_2'
# for i in os.listdir(root_path):
#   if i.rfind('costa')!=-1:
#     save_path = join(save_dir, i, 'test_data')
#     # os.makedirs(save_path, exist_ok=True)
#     shutil.copytree(join(root_path, i, 'test_data'), save_path, dirs_exist_ok=True)

In [12]:
for root, dirs, files  in os.walk('/data1/liuyang/stent/interior/data/corr_data/data/costa/voxel_0.2/trun_data_yes_fdk'):
  for f in files:
    if(not f.endswith('0-60.nii.gz')):
      os.remove(join(root, f))

### 测试一下运行时间

但是对于我这个任务来说，不可以只对roi内的部分做投影

In [2]:
def set_geo(DSD  =1250, 
            DSO = 750,
            nDetector = np.array([512,512]),
            dDetector = np.array([0.3, 0.3]), 
            nVoxel = np.array([248, 510, 636]),
            dVoxel = np.array([0.3, 0.3, 0.6]),
            geo_type='cone'):
    # Lets create a geometry object
    geo = tigre.geometry()
    # VARIABLE                                   DESCRIPTION                    UNITS
    # -------------------------------------------------------------------------------------
    # Distances
    geo.DSD = DSD  # Distance Source Detector      (mm)
    geo.DSO = DSO  # Distance Source Origin        (mm)
    # Detector parameters
    geo.nDetector = nDetector  # number of pixels              (px)
    geo.dDetector =dDetector  # size of each pixel            (mm)
    geo.sDetector = geo.nDetector * geo.dDetector  # total size of the detector    (mm)
    # Image parameters
    geo.nVoxel = nVoxel  # number of voxels              (vx)
    geo.dVoxel= dVoxel # size of each voxel            (mm)
    geo.sVoxel = geo.dVoxel * geo.nVoxel  # total size of the image       (mm)
    
    # Offsets
    geo.offOrigin = np.array([0, 0, 0])  # Offset of image from origin   (mm)
    # geo.offOrigin = geo.sVoxel /2 

    geo.offDetector = np.array([0, 0])  # Offset of Detector            (mm)
    # geo.offDetector = geo.sDetector / 2

    geo.accuracy = 0.5  # Variable to define accuracy of
    geo.COR = 0  # y direction displacement for
    geo.rotDetector = np.array([0, 0, 0])  # Rotation of the detector, by
    # geo.mode = "cone"  # Or 'parallel'. Geometry type.
    geo.mode = geo_type  # Or 'parallel'. Geometry type.
    return geo

In [3]:
import time

In [12]:
## 测试一下所花时间
image_nib = nib.load('/data1/liuyang/stent/interior/data/costa/111/resample_image.nii.gz')
image_nib_data = image_nib.get_fdata().astype(np.float32)
mask_nib = nib.load('/data1/liuyang/stent/interior/data/corr_data/data/costa/voxel_0.2/src_data/radius-64/corr_vessel/train_data/vol_gt_mask/ICBM_010_sample_00.nii.gz')
mask_nib_data = np.round(mask_nib.get_fdata()).astype(np.int16)

z_dim_max, x_dim_max, y_dim_max = image_nib.shape[0], image_nib.shape[1], image_nib.shape[2]
[x_voxel_size, y_voxel_size, z_voxel_size] = [round(j, 4) for j in  image_nib.header['pixdim'][1:4]]

## 设置投影几何
geo = set_geo(nVoxel=np.array([z_dim_max, x_dim_max, y_dim_max]), 
        dVoxel=np.array([z_voxel_size, x_voxel_size, y_voxel_size]),
        nDetector=np.array([1024*2, 1024*2]),
        dDetector=np.array([x_voxel_size, y_voxel_size]))


In [5]:
angles = np.array([np.linspace(0, 2*np.pi, 360)]).reshape(360, )

In [13]:
image_nib_data.shape

(688, 918, 696)

In [14]:
## 先投影，再去roi
start = time.time()
proj_01 = tigre.Ax(image_nib_data, geo, angles)
end = time.time()
print(f"耗时: {end - start:.6f} 秒")

耗时: 23.176922 秒


In [15]:
## 先取roi，再投影
start = time.time()
roi_data = image_nib_data * mask_nib_data
proj_01 = tigre.Ax(roi_data, geo, angles)
end = time.time()
print(f"耗时: {end - start:.6f} 秒")

耗时: 23.286210 秒


In [8]:
len(np.where(roi_data>0)[0])

57956

In [16]:
len(np.where(image_nib_data>0)[0])

321394668

In [10]:
len(np.where(mask_nib_data>0)[0])

1097917

## 其他

In [ ]:
# "/data1/liuyang/stent/interior/data/corr_data/data/costa_test_2/voxel_0.2/trun_data/radius-64/corr_stent/test_data/90_2-fdkNoFilter/predict/ICBM-001-sample-00-10-90-180.nii.gz".rfind("90-180")

161